In [ ]:
# Ignore annoying warning from astropy fits reader about wrong units in the FITS files
%env PYTHONWARNINGS=ignore

In [ ]:
import os
from itertools import chain
from pathlib import Path

import pyarrow as pa
from astropy.io import fits
from hats_import import CollectionArguments, pipeline_with_client
from hats_import.catalog.file_readers import InputReader, FitsReader
from dask.distributed import Client

In [ ]:
RAW_ROOT = Path('/sdf/data/rubin/cosmicdata/lsdb/test_data_transfer/hats-publish/legacysurvey_dr11/raw_dr11/')
HATS_ROOT = Path("/sdf/data/rubin/cosmicdata/lsdb/test_data_transfer/hats-publish/legacysurvey_dr11/") / "hats"

DR = '11.0'
NORTH_SOUTH = "south"

HATS_ROOT.mkdir(parents=True, exist_ok=True)
CAT_NAME = f"legacysurvey_dr{DR}_{NORTH_SOUTH}"
OUTPUT_DIR = HATS_ROOT / CAT_NAME

INPUT_ROOT = RAW_ROOT / NORTH_SOUTH / "sweep"
MAIN_INPUT_DIR = INPUT_ROOT / DR
assert MAIN_INPUT_DIR.exists(), str(MAIN_INPUT_DIR)

In [ ]:
some_main_input_path = next(MAIN_INPUT_DIR.glob("*.fits"))
main_cat_columns = fits.open(some_main_input_path)[1].columns.names

In [ ]:
class LegacySurveyReader(InputReader):
    def __init__(self, chunksize=100_000, supplementary_catalogs=None):
        self.chunksize = chunksize
        self.supplementary_catalogs = supplementary_catalogs
        self.fits_reader = FitsReader(chunksize=self.chunksize)
    
    def read(self, input_file, read_columns=None):
        if read_columns is not None or self.supplementary_catalogs is None:
            yield from self.fits_reader.read(input_file, read_columns=read_columns)
            return
        
        main_gen = self.fits_reader.read(input_file, read_columns=read_columns)
        suppl_gens = []
        for dir_path, file_suffix in self.supplementary_catalogs:
            suppl_catalog_fname = f'{input_file.stem}{file_suffix}.fits'
            suppl_catalog_path = dir_path / suppl_catalog_fname
            suppl_gens.append(
                self.fits_reader.read(suppl_catalog_path, read_columns=read_columns),
            )

        # Concat tables
        for main_table, *suppl_tables in zip(main_gen, *suppl_gens):
            for suppl_table_len in map(len, suppl_tables):
                assert suppl_table_len == len(main_table)
            # Remove suppl columns which are in main table or in a previous suppl catalog
            seen = set(main_table.schema.names)
            for i, suppl in enumerate(suppl_tables):
                suppl_tables[i] = suppl.drop_columns([n for n in suppl.schema.names if n in seen])
                seen.update(suppl_tables[i].schema.names)
            concat_table = pa.Table.from_arrays(
                list(main_table.columns) + list(chain.from_iterable(t.columns for t in suppl_tables)),
                names=list(main_table.schema.names) + list(chain.from_iterable(t.schema.names for t in suppl_tables)),
            )

            # Nest LC_* columns, cast to "normal" list from fixed
            lc_names = [n for n in concat_table.column_names if n.startswith("LC_")]
            c = concat_table.select(lc_names).combine_chunks()
            fields = [n.removeprefix("LC_") for n in lc_names]
            lc_array = pa.StructArray.from_arrays(
                [c.column(n).chunk(0).cast(pa.list_(c.column(n).type.value_type)) for n in lc_names],
                names=fields,
            )
            concat_table = concat_table.drop_columns(lc_names).append_column("LC", lc_array)
            
            yield concat_table


In [ ]:
args = CollectionArguments(
    output_artifact_name=CAT_NAME,
    output_path=HATS_ROOT,
    skymap_alt_orders=[4, 6, 8],
    create_thumbnail=True,
    create_per_partition_stats=True,
    create_skymap_png=True,
    create_partition_info_png=True,
    create_summary_html=True,    
).catalog(
    input_file_list=sorted(MAIN_INPUT_DIR.glob("*.fits")),
    # input_file_list=[MAIN_INPUT_DIR / 'sweep-000m005-005p000.fits'],
    pixel_threshold=100_000,
    file_reader=LegacySurveyReader(
        chunksize=10_000,
        supplementary_catalogs=[
            # folder path, file suffix
            (INPUT_ROOT / f"{DR}-extra", '-ex'),
            (INPUT_ROOT / f"{DR}-lightcurves", '-lc'),
            (INPUT_ROOT / f"{DR}-photo-z", '-pz'),
        ],
    ),
    ra_column="RA",
    dec_column="DEC",
    sort_columns="OBJID",
    addl_hats_properties={
        'hats_cols_default': main_cat_columns,
    },
).add_margin(
    margin_threshold=5.0,
    is_default=True,
).add_margin(
    margin_threshold=60.0,
    is_default=False,
).add_index(
    indexing_column="OBJID",
    include_healpix_29=True,
    include_order_pixel=True,
    drop_duplicates=False,
)

with Client(n_workers=16, threads_per_worker=1, memory_limit=f"64GB") as client:
    display(client)
    pipeline_with_client(args, client)

In [ ]:
import lsdb

cat = lsdb.open_catalog(HATS_ROOT / CAT_NAME, columns='all')
cat